# Tinh chỉnh PaddleOCR đọc biển số Việt Nam — Nhóm 08

Huấn luyện tiếp mô hình **nhận dạng chữ** `en_PP-OCRv5_mobile_rec` (mô hình hệ thống đang dùng) trên ảnh **từng dòng** biển số đã gán nhãn tay.

**Chuẩn bị trên máy (một lần mỗi khi có thêm nhãn):**
```
python scripts/export_paddle_rec.py --zip
```
→ tạo `data/paddle_rec.zip` (chỉ biển train/val đã kiểm tra; biển tập test KHÔNG được xuất). Tải file zip này lên **thư mục gốc My Drive**.

**Trên Colab:** menu *Thời gian chạy → Thay đổi loại thời gian chạy → T4 GPU*, rồi chạy lần lượt từng ô.

**Kết quả:** file `paddle_rec.zip` (mô hình suy luận). Giải nén vào `weights/paddle_rec/` trong dự án → hệ thống tự dùng mô hình mới. Sau đó đo lại trên máy: `python scripts/eval_ocr.py` (chỉ đo trên tập test).

## 1. Cài PaddlePaddle GPU + PaddleOCR

In [ ]:
import shutil, subprocess
# Bước kiểm tra: phải có GPU (menu Thời gian chạy -> Thay đổi loại thời gian chạy -> T4 GPU)
if shutil.which('nvidia-smi') is None:
    raise SystemExit('CHƯA CÓ GPU: vào menu Thời gian chạy (Runtime) -> Thay đổi loại thời gian chạy -> chọn T4 GPU -> Lưu, '
                     'rồi chạy lại từ ô này. Nếu không chọn được T4: hết hạn mức GPU miễn phí, đợi vài giờ hoặc dùng tài khoản khác.')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv'], capture_output=True, text=True).stdout)
# Lệnh cài theo trang cài đặt chính thức của PaddlePaddle (bản CUDA 12.6)
!pip install -q paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!git clone -q --depth 1 https://github.com/PaddlePaddle/PaddleOCR /content/PaddleOCR
%cd /content/PaddleOCR
!pip install -q -r requirements.txt
!pip install -q paddleocr
import paddle; paddle.utils.run_check()

## 2. Lấy dữ liệu từ Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile
ZIP = '/content/drive/MyDrive/paddle_rec.zip'
assert os.path.exists(ZIP), 'Chưa thấy paddle_rec.zip ở thư mục gốc My Drive (chạy scripts/export_paddle_rec.py --zip trên máy rồi tải lên)'
DATA = '/content/train_data'
zipfile.ZipFile(ZIP).extractall(DATA)
for s in ('train', 'val'):
    n = sum(1 for l in open(f'{DATA}/{s}.txt', encoding='utf-8') if l.strip())
    print(s, n, 'ảnh dòng chữ')
!head -3 /content/train_data/train.txt

## 3. Kiểm tra cấu hình + bảng ký tự (báo lỗi sớm nếu kho mã PaddleOCR đổi cấu trúc)

In [ ]:
import yaml
CFG = 'configs/rec/PP-OCRv5/multi_language/en_PP-OCRv5_mobile_rec.yaml'
assert os.path.exists(CFG), f'Không thấy {CFG} - xem lại thư mục configs/rec/PP-OCRv5/multi_language/'
cfg = yaml.safe_load(open(CFG, encoding='utf-8'))
DICT = cfg['Global']['character_dict_path']
assert os.path.exists(DICT), f'Không thấy bảng ký tự {DICT}'
chars = set(open(DICT, encoding='utf-8').read().split('
'))
labels = [l.rstrip('
').split('	') for s in ('train', 'val') for l in open(f'{DATA}/{s}.txt', encoding='utf-8') if l.strip()]
missing = {c for _, lab in labels for c in lab if c not in chars}
assert not missing, f'Ký tự không có trong bảng: {missing}'
assert all(os.path.exists(os.path.join(DATA, p)) for p, _ in labels[:50]), 'Đường dẫn ảnh trong file nhãn phải tương đối với thư mục dữ liệu'
print('Cấu hình OK:', CFG)
print('Dataset train:', cfg['Train']['dataset']['name'], '| loader:', cfg['Train']['loader'], '| sampler:', cfg['Train'].get('sampler'))

## 4. Tải trọng số gốc (bản dùng để huấn luyện tiếp)

In [ ]:
!wget -q -nc https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/en_PP-OCRv5_mobile_rec_pretrained.pdparams
!ls -la en_PP-OCRv5_mobile_rec_pretrained.pdparams

## 5. Huấn luyện

Dữ liệu nhỏ (vài trăm – vài nghìn ảnh dòng) nên giảm batch và tốc độ học so với mặc định (batch 128, lr 0.0005), đánh giá thường xuyên.
Mô hình tốt nhất trên tập val được lưu thành `best_accuracy`. Nếu báo lỗi khoá cấu hình (`KeyError`), xem lại tên khoá trong ô 3 (dòng `loader` / `sampler`).

In [ ]:
EPOCHS, BS, LR = 60, 64, 0.0002
sampler_opt = f'Train.sampler.first_bs={BS}' if cfg['Train'].get('sampler') else ''
!python tools/train.py -c {CFG} -o     Global.pretrained_model=./en_PP-OCRv5_mobile_rec_pretrained.pdparams     Global.save_model_dir=./output/plate_rec Global.epoch_num={EPOCHS}     Global.eval_batch_step=[0,50] Global.save_epoch_step=1000 Global.distributed=False     Train.dataset.data_dir={DATA}/ Train.dataset.label_file_list=[{DATA}/train.txt]     Eval.dataset.data_dir={DATA}/ Eval.dataset.label_file_list=[{DATA}/val.txt]     Train.loader.batch_size_per_card={BS} Eval.loader.batch_size_per_card={BS}     Train.loader.num_workers=2 Eval.loader.num_workers=2     Optimizer.lr.learning_rate={LR} Optimizer.lr.warmup_epoch=2 {sampler_opt}

In [ ]:
!grep -E "best metric|acc:" output/plate_rec/train.log | tail -5

## 6. Xuất mô hình suy luận + kiểm tra đọc trên tập val

In [ ]:
!python tools/export_model.py -c {CFG} -o Global.pretrained_model=./output/plate_rec/best_accuracy Global.save_inference_dir=/content/paddle_rec
!ls /content/paddle_rec
from paddleocr import TextRecognition
import re
for name, kw in (('Gốc', {}), ('Tinh chỉnh', {'model_dir': '/content/paddle_rec'})):
    m = TextRecognition(model_name='en_PP-OCRv5_mobile_rec', **kw)
    val = [l.rstrip('
').split('	') for l in open(f'{DATA}/val.txt', encoding='utf-8') if l.strip()]
    ok = sum(re.sub('[^A-Z0-9]', '', next(iter(m.predict(os.path.join(DATA, p))))['rec_text'].upper()) == lab for p, lab in val)
    print(f'{name}: đọc đúng {ok}/{len(val)} dòng val ({ok/len(val):.1%})')

## 7. Tải mô hình về máy
Giải nén vào `weights/paddle_rec/` (trong đó có `inference.pdiparams`), khởi động lại web, rồi đo: `python scripts/eval_ocr.py`.

In [ ]:
import shutil
shutil.make_archive('/content/paddle_rec', 'zip', '/content/paddle_rec')
shutil.copy('/content/paddle_rec.zip', '/content/drive/MyDrive/paddle_rec_model.zip')
from google.colab import files
files.download('/content/paddle_rec.zip')